# 2048 DQN — Base vs Enhance Head-to-Head

This notebook runs **both the original base model** and a **stronger enhanced model** for OpenSpiel 2048, then compares them on the **same validation seeds** and **same test seeds**.

## What this notebook is built to do
1. Reproduce the **baseline spirit** of `DQN_for_2048_game.ipynb`.
2. Train a stronger **enhanced** agent that is intentionally configured to have a high chance of outperforming the base.
3. Produce a **clean comparison table** for reporting:
   - Greedy return
   - Validation mean return
   - Test mean return
   - Test std
   - P(max tile >= 256)
   - P(max tile >= 512)
4. Provide **plots and detailed evaluation** so the report is not based on a single lucky rollout.

## Important note on fairness
This notebook supports two modes:

- **Fair comparison**: both models use the same training budget.
- **Maximize enhanced**: the enhanced model gets a larger budget and a stronger architecture so it is more likely to outperform the base.

By default, this notebook is set to **maximize the enhanced model**, because the practical goal here is to make the enhanced agent clearly stronger than the base. If your instructor requires a pure ablation, switch `FAIR_COMPARE = True` in the config cell.


In [ ]:
# Colab / notebook setup
!python -V
!pip -q install --upgrade pip
!pip -q install open-spiel torch matplotlib imageio tqdm pandas


In [ ]:
import os
import math
import random
import re
from copy import deepcopy
from dataclasses import dataclass
from collections import deque, namedtuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pyspiel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)


## 1. Inspect OpenSpiel 2048


In [ ]:
game = pyspiel.load_game("2048")
state = game.new_initial_state()

game_type = game.get_type()
short_name = game_type.short_name if hasattr(game_type, "short_name") else "2048"

print("Registered game:", short_name)
print("Num distinct actions:", game.num_distinct_actions())
print("Observation tensor size:", game.observation_tensor_size())
print("Initial state is chance node:", state.is_chance_node())
print()
print(state)


## 2. Helpers and thin wrapper

The wrapper is intentionally kept very close to the base notebook:
- auto-resolve chance nodes,
- expose `reset()` and `step(action)`,
- use **delta in cumulative return** as reward,
- keep **legal-action masking** outside the environment.

This lets us compare the learner more cleanly.


In [ ]:
def extract_obs(state, player_id=0):
    """Return a flat float32 observation vector for the player."""
    for fn_name, args in [
        ("observation_tensor", (player_id,)),
        ("observation_tensor", tuple()),
        ("information_state_tensor", (player_id,)),
        ("information_state_tensor", tuple()),
    ]:
        fn = getattr(state, fn_name, None)
        if fn is None:
            continue
        try:
            obs = fn(*args)
            obs = np.asarray(obs, dtype=np.float32).reshape(-1)
            return obs
        except TypeError:
            pass
    raise RuntimeError("Could not extract an observation tensor from state.")

def legal_actions(state, player_id=0):
    try:
        return list(state.legal_actions(player_id))
    except TypeError:
        return list(state.legal_actions())

def sample_chance_action(state, rng):
    outcomes = state.chance_outcomes()
    actions, probs = zip(*outcomes)
    idx = rng.choice(len(actions), p=np.asarray(probs, dtype=np.float64))
    return actions[idx]

def auto_resolve_chance_nodes(state, rng):
    while state.is_chance_node() and not state.is_terminal():
        a = sample_chance_action(state, rng)
        state.apply_action(a)
    return state

def state_return(state, player_id=0):
    vals = state.returns()
    return float(vals[player_id]) if len(vals) > player_id else 0.0

def state_reward(state, player_id=0):
    vals = state.rewards()
    return float(vals[player_id]) if len(vals) > player_id else 0.0

def parse_board_numbers(state):
    txt = str(state)
    nums = [int(x) for x in re.findall(r"\d+", txt)]
    if len(nums) >= 16:
        nums = nums[-16:]
        return np.array(nums, dtype=np.int64).reshape(4, 4)
    return None

class OpenSpiel2048Env:
    def __init__(self, seed=42):
        self.game = pyspiel.load_game("2048")
        self.player_id = 0
        self.num_actions = self.game.num_distinct_actions()
        self.obs_dim = self.game.observation_tensor_size()
        self.rng = np.random.default_rng(seed)
        self.state = None

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.game.new_initial_state()
        auto_resolve_chance_nodes(self.state, self.rng)
        return extract_obs(self.state, self.player_id)

    def step(self, action):
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        if self.state.is_terminal():
            raise RuntimeError("Episode already ended. Call reset().")

        legal = legal_actions(self.state, self.player_id)
        if action not in legal:
            raise ValueError(f"Illegal action {action}. Legal actions: {legal}")

        prev_return = state_return(self.state, self.player_id)

        self.state.apply_action(int(action))
        auto_resolve_chance_nodes(self.state, self.rng)

        next_obs = extract_obs(self.state, self.player_id) if not self.state.is_terminal() else np.zeros(self.obs_dim, dtype=np.float32)
        new_return = state_return(self.state, self.player_id)

        reward = new_return - prev_return
        done = self.state.is_terminal()
        info = {
            "legal_actions": legal_actions(self.state, self.player_id) if not done else [],
            "state_return": new_return,
            "state_reward_raw": state_reward(self.state, self.player_id),
            "board": parse_board_numbers(self.state),
            "state_text": str(self.state),
        }
        return next_obs, float(reward), done, info

    def legal_actions(self):
        if self.state is None or self.state.is_terminal():
            return []
        return legal_actions(self.state, self.player_id)

    def render(self):
        if self.state is None:
            print("<env not reset>")
        else:
            print(self.state)

test_env = OpenSpiel2048Env(seed=123)
obs = test_env.reset()
print("Observation shape:", obs.shape)
print("Initial legal actions:", test_env.legal_actions())
print("Initial board:")
print(parse_board_numbers(test_env.state))


## 3. Replay buffers, masking, and model classes


In [ ]:
Transition = namedtuple(
    "Transition",
    ["obs", "action", "reward", "next_obs", "done", "discount", "legal_mask", "next_legal_mask"],
)

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def add(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

class NStepTransitionCollector:
    def __init__(self, n_step, gamma):
        self.n_step = n_step
        self.gamma = gamma
        self.queue = deque()

    def reset(self):
        self.queue.clear()

    def __len__(self):
        return len(self.queue)

    def _build_transition_from_front(self):
        obs, action, _, _, _, legal_mask, _ = self.queue[0]
        total_reward = 0.0
        discount = 1.0
        next_obs = self.queue[-1][3]
        done = self.queue[-1][4]
        next_legal_mask = self.queue[-1][6]

        for _, _, reward_i, next_obs_i, done_i, _, next_legal_mask_i in self.queue:
            total_reward += discount * float(reward_i)
            next_obs = next_obs_i
            done = done_i
            next_legal_mask = next_legal_mask_i
            if done_i:
                return Transition(
                    obs, action, float(total_reward), next_obs, True, 0.0, legal_mask, next_legal_mask
                )
            discount *= self.gamma

        return Transition(
            obs, action, float(total_reward), next_obs, done, float(discount), legal_mask, next_legal_mask
        )

    def add(self, obs, action, reward, next_obs, done, legal_mask, next_legal_mask):
        self.queue.append((obs, action, reward, next_obs, done, legal_mask, next_legal_mask))
        ready = []
        if len(self.queue) >= self.n_step:
            ready.append(self._build_transition_from_front())
            self.queue.popleft()
        if done:
            while self.queue:
                ready.append(self._build_transition_from_front())
                self.queue.popleft()
        return ready

def make_legal_mask(num_actions, legal_actions_list):
    mask = np.zeros(num_actions, dtype=np.float32)
    if len(legal_actions_list) > 0:
        mask[legal_actions_list] = 1.0
    return mask

@torch.no_grad()
def masked_greedy_action(q_net, obs, legal_actions_list, num_actions, epsilon=0.0, device=DEVICE):
    if len(legal_actions_list) == 0:
        raise RuntimeError("No legal actions available.")
    if random.random() < epsilon:
        return random.choice(legal_actions_list)

    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    q = q_net(obs_t).squeeze(0)

    legal_mask = torch.zeros(num_actions, dtype=torch.bool, device=device)
    legal_mask[legal_actions_list] = True

    q_masked = q.masked_fill(~legal_mask, -1e9)
    return int(torch.argmax(q_masked).item())

class BaseQNetwork(nn.Module):
    def __init__(self, obs_dim, num_actions, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )

    def forward(self, x):
        return self.net(x)

class ResidualMLPBlock(nn.Module):
    def __init__(self, dim, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.ReLU()

    def forward(self, x):
        h = self.fc1(x)
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout(h)
        h = self.fc2(h)
        h = self.norm2(h)
        return self.act(x + h)

class DuelingResidualQNetwork(nn.Module):
    def __init__(self, obs_dim, num_actions, hidden_dim=256, num_blocks=2, dropout=0.0):
        super().__init__()
        self.input = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[
            ResidualMLPBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.adv_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.input(x)
        h = self.blocks(h)
        value = self.value_head(h)
        adv = self.adv_head(h)
        q = value + adv - adv.mean(dim=1, keepdim=True)
        return q


## 4. Experiment configuration

The **base** config stays very close to the original notebook.  
The **enhanced** config is intentionally stronger:
- dueling residual MLP,
- Double DQN,
- Huber loss,
- 3-step returns,
- longer training budget by default.

If you need a stricter apples-to-apples comparison, set `FAIR_COMPARE = True`.


In [ ]:
# ---------- Global evaluation protocol ----------
GLOBAL_SEED = 7
VAL_SEEDS = list(range(10_000, 10_050))     # fixed validation seeds
TEST_SEEDS = list(range(20_000, 20_100))    # fixed test seeds
REPORT_GREEDY_SEED = 999
MAX_STEPS_PER_EPISODE = 5_000

# ---------- Comparison mode ----------
FAIR_COMPARE = False   # True = same budget for base and enhance
FAST_SMOKE_TEST = False  # True = tiny run to validate the notebook quickly

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_global_seed(GLOBAL_SEED)

BASE_CONFIG = {
    "name": "Base",
    "model_type": "base_mlp",
    "hidden_dim": 256,
    "num_res_blocks": 0,
    "dropout": 0.0,
    "double_dqn": False,
    "loss_type": "mse",
    "n_step": 1,
    "num_episodes": 3_000,
    "buffer_size": 100_000,
    "batch_size": 128,
    "gamma": 0.99,
    "lr": 1e-3,
    "target_sync_every": 250,
    "learn_start": 1_000,
    "learn_every": 4,
    "eps_start": 1.0,
    "eps_end": 0.05,
    "eps_decay_steps": 20_000,
    "grad_clip": 10.0,
    "eval_every": 100,
    "train_seed": 7,
}

ENHANCE_CONFIG = {
    "name": "Enhance",
    "model_type": "dueling_residual",
    "hidden_dim": 256,
    "num_res_blocks": 2,
    "dropout": 0.0,
    "double_dqn": True,
    "loss_type": "huber",
    "n_step": 3,
    "num_episodes": 12_000,
    "buffer_size": 150_000,
    "batch_size": 128,
    "gamma": 0.99,
    "lr": 5e-4,
    "target_sync_every": 500,
    "learn_start": 2_000,
    "learn_every": 4,
    "eps_start": 1.0,
    "eps_end": 0.05,
    "eps_decay_steps": 60_000,
    "grad_clip": 10.0,
    "eval_every": 100,
    "train_seed": 7,
}

if FAIR_COMPARE:
    ENHANCE_CONFIG["num_episodes"] = BASE_CONFIG["num_episodes"]
    ENHANCE_CONFIG["buffer_size"] = BASE_CONFIG["buffer_size"]
    ENHANCE_CONFIG["eps_decay_steps"] = BASE_CONFIG["eps_decay_steps"]

if FAST_SMOKE_TEST:
    BASE_CONFIG["num_episodes"] = 50
    BASE_CONFIG["eval_every"] = 10
    ENHANCE_CONFIG["num_episodes"] = 80
    ENHANCE_CONFIG["eval_every"] = 10
    VAL_SEEDS = list(range(10_000, 10_010))
    TEST_SEEDS = list(range(20_000, 20_020))

print("FAIR_COMPARE =", FAIR_COMPARE)
print("FAST_SMOKE_TEST =", FAST_SMOKE_TEST)
print("Base config:")
print(BASE_CONFIG)
print("Enhance config:")
print(ENHANCE_CONFIG)


## 5. Shared training and evaluation functions


In [ ]:
def build_model(config, obs_dim, num_actions):
    if config["model_type"] == "base_mlp":
        return BaseQNetwork(obs_dim, num_actions, hidden_dim=config["hidden_dim"]).to(DEVICE)
    if config["model_type"] == "dueling_residual":
        return DuelingResidualQNetwork(
            obs_dim, num_actions,
            hidden_dim=config["hidden_dim"],
            num_blocks=config["num_res_blocks"],
            dropout=config["dropout"],
        ).to(DEVICE)
    raise ValueError(f"Unknown model_type: {config['model_type']}")

def epsilon_by_step(step, config):
    frac = min(1.0, step / config["eps_decay_steps"])
    return config["eps_start"] + frac * (config["eps_end"] - config["eps_start"])

def dqn_update(batch, q_net, target_net, optimizer, config):
    obs = torch.tensor(np.asarray(batch.obs), dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(batch.action, dtype=torch.int64, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE)
    next_obs = torch.tensor(np.asarray(batch.next_obs), dtype=torch.float32, device=DEVICE)
    dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE)
    discounts = torch.tensor(batch.discount, dtype=torch.float32, device=DEVICE)

    next_legal_mask = torch.tensor(np.asarray(batch.next_legal_mask), dtype=torch.bool, device=DEVICE)

    q_values = q_net(obs)
    q_sa = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        if config["double_dqn"]:
            next_q_online = q_net(next_obs)
            next_q_online = next_q_online.masked_fill(~next_legal_mask, -1e9)
            next_actions = torch.argmax(next_q_online, dim=1, keepdim=True)
            next_q = target_net(next_obs).gather(1, next_actions).squeeze(1)
        else:
            next_q_all = target_net(next_obs)
            next_q_all = next_q_all.masked_fill(~next_legal_mask, -1e9)
            next_q = next_q_all.max(dim=1).values

        next_q = torch.where(dones > 0.5, torch.zeros_like(next_q), next_q)
        target = rewards + discounts * next_q

    if config["loss_type"] == "huber":
        loss = F.smooth_l1_loss(q_sa, target)
    elif config["loss_type"] == "mse":
        loss = F.mse_loss(q_sa, target)
    else:
        raise ValueError(f"Unknown loss_type: {config['loss_type']}")

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), config["grad_clip"])
    optimizer.step()

    return float(loss.item())

@torch.no_grad()
def evaluate_policy(model, seeds, num_actions, max_steps=MAX_STEPS_PER_EPISODE):
    returns = []
    lengths = []
    max_tiles = []

    for seed in seeds:
        env = OpenSpiel2048Env(seed=seed)
        obs = env.reset(seed=seed)
        done = False
        ep_return = 0.0
        ep_len = 0
        last_board = None

        while not done and ep_len < max_steps:
            legal = env.legal_actions()
            action = masked_greedy_action(model, obs, legal, num_actions, epsilon=0.0, device=DEVICE)
            obs, reward, done, info = env.step(action)
            ep_return += reward
            ep_len += 1
            if info.get("board") is not None:
                last_board = info["board"]

        returns.append(float(ep_return))
        lengths.append(int(ep_len))
        max_tiles.append(int(last_board.max()) if last_board is not None else 0)

    returns = np.asarray(returns, dtype=np.float32)
    lengths = np.asarray(lengths, dtype=np.float32)
    max_tiles = np.asarray(max_tiles, dtype=np.int64)

    return {
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_length": float(np.mean(lengths)),
        "mean_max_tile": float(np.mean(max_tiles)),
        "returns": returns,
        "lengths": lengths,
        "max_tiles": max_tiles,
    }

def summarize_tiles(max_tiles):
    max_tiles = np.asarray(max_tiles)
    summary = {}
    for level in [64, 128, 256, 512, 1024, 2048]:
        summary[level] = float(np.mean(max_tiles >= level))
    return summary

@torch.no_grad()
def single_greedy_rollout(model, seed, num_actions, max_steps=MAX_STEPS_PER_EPISODE):
    env = OpenSpiel2048Env(seed=seed)
    obs = env.reset(seed=seed)
    done = False
    ep_return = 0.0
    rollout = []
    while not done and len(rollout) < max_steps:
        legal = env.legal_actions()
        action = masked_greedy_action(model, obs, legal, num_actions, epsilon=0.0, device=DEVICE)
        next_obs, reward, done, info = env.step(action)
        rollout.append({
            "action": action,
            "reward": reward,
            "legal_actions": legal,
            "board": info["board"],
            "state_text": info["state_text"],
        })
        obs = next_obs
        ep_return += reward
    last_board = rollout[-1]["board"] if rollout and rollout[-1]["board"] is not None else None
    return {
        "greedy_return": float(ep_return),
        "rollout_length": len(rollout),
        "max_tile": int(np.max(last_board)) if last_board is not None else 0,
        "last_board": last_board,
        "rollout": rollout,
    }

def print_eval_summary(tag, metrics):
    tile_summary = summarize_tiles(metrics["max_tiles"])
    print(
        f"{tag} | mean_return={metrics['mean_return']:.1f} | std={metrics['std_return']:.1f} | "
        f"mean_len={metrics['mean_length']:.1f} | mean_max_tile={metrics['mean_max_tile']:.1f} | "
        f">=256={100*tile_summary[256]:.1f}% | >=512={100*tile_summary[512]:.1f}%"
    )


## 6. Training routine

The enhanced model is intentionally given a stronger learning setup so it has a realistic chance to beat the base by a clear margin. If your assignment requires a stricter ablation, toggle `FAIR_COMPARE = True`.


In [ ]:
def train_variant(config):
    set_global_seed(config["train_seed"])

    env = OpenSpiel2048Env(seed=config["train_seed"])
    obs_dim = env.obs_dim
    num_actions = env.num_actions

    q_net = build_model(config, obs_dim, num_actions)
    target_net = build_model(config, obs_dim, num_actions)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=config["lr"])
    replay = ReplayBuffer(config["buffer_size"])
    n_step_collector = NStepTransitionCollector(config["n_step"], config["gamma"])

    episode_returns = []
    episode_lengths = []
    loss_history = []
    val_history = []

    global_step = 0
    best_eval_mean = -float("inf")
    best_state_dict = None

    progress = tqdm(range(1, config["num_episodes"] + 1), desc=f"Training {config['name']}")
    for episode in progress:
        obs = env.reset(seed=config["train_seed"] + episode)
        done = False
        ep_return = 0.0
        ep_len = 0
        n_step_collector.reset()

        while not done and ep_len < MAX_STEPS_PER_EPISODE:
            eps = epsilon_by_step(global_step, config)
            legal = env.legal_actions()
            legal_mask = make_legal_mask(num_actions, legal)

            action = masked_greedy_action(
                q_net=q_net,
                obs=obs,
                legal_actions_list=legal,
                num_actions=num_actions,
                epsilon=eps,
                device=DEVICE,
            )

            next_obs, reward, done, info = env.step(action)
            next_legal = info["legal_actions"] if not done else []
            next_legal_mask = make_legal_mask(num_actions, next_legal)

            ready_transitions = n_step_collector.add(
                obs, action, reward, next_obs, done, legal_mask, next_legal_mask
            )
            for tr in ready_transitions:
                replay.add(*tr)

            obs = next_obs
            ep_return += reward
            ep_len += 1
            global_step += 1

            if len(replay) >= max(config["learn_start"], config["batch_size"]) and global_step % config["learn_every"] == 0:
                batch = replay.sample(config["batch_size"])
                loss = dqn_update(batch, q_net, target_net, optimizer, config)
                loss_history.append(loss)

            if global_step % config["target_sync_every"] == 0:
                target_net.load_state_dict(q_net.state_dict())

        episode_returns.append(ep_return)
        episode_lengths.append(ep_len)

        if episode % config["eval_every"] == 0:
            metrics = evaluate_policy(q_net, VAL_SEEDS, num_actions)
            tile_summary = summarize_tiles(metrics["max_tiles"])
            val_history.append({
                "episode": episode,
                "mean_return": metrics["mean_return"],
                "std_return": metrics["std_return"],
                "mean_length": metrics["mean_length"],
                "mean_max_tile": metrics["mean_max_tile"],
                "tile_summary": tile_summary,
            })
            if metrics["mean_return"] > best_eval_mean:
                best_eval_mean = metrics["mean_return"]
                best_state_dict = {k: v.detach().cpu().clone() for k, v in q_net.state_dict().items()}

            progress.set_postfix({
                "val_mean": f"{metrics['mean_return']:.0f}",
                "best": f"{best_eval_mean:.0f}",
                "eps": f"{eps:.3f}",
            })

    if best_state_dict is not None:
        q_net.load_state_dict(best_state_dict)
        target_net.load_state_dict(best_state_dict)

    val_metrics = evaluate_policy(q_net, VAL_SEEDS, num_actions)
    test_metrics = evaluate_policy(q_net, TEST_SEEDS, num_actions)
    greedy = single_greedy_rollout(q_net, REPORT_GREEDY_SEED, num_actions)

    result = {
        "config": deepcopy(config),
        "obs_dim": obs_dim,
        "num_actions": num_actions,
        "q_net": q_net,
        "target_net": target_net,
        "episode_returns": episode_returns,
        "episode_lengths": episode_lengths,
        "loss_history": loss_history,
        "val_history": val_history,
        "best_val_mean_return": best_eval_mean,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_tile_summary": summarize_tiles(test_metrics["max_tiles"]),
        "greedy": greedy,
    }
    return result


## 7. Run base and enhanced models


In [ ]:
base_result = train_variant(BASE_CONFIG)
enhance_result = train_variant(ENHANCE_CONFIG)


## 8. Primary comparison table


In [ ]:
def make_comparison_row(result):
    row = {
        "Model": result["config"]["name"],
        "Greedy return": result["greedy"]["greedy_return"],
        "Validation mean return": result["val_metrics"]["mean_return"],
        "Test mean return": result["test_metrics"]["mean_return"],
        "Std": result["test_metrics"]["std_return"],
        "P(max tile >= 256)": result["test_tile_summary"][256],
        "P(max tile >= 512)": result["test_tile_summary"][512],
        "Mean max tile": result["test_metrics"]["mean_max_tile"],
        "Rollout length": result["greedy"]["rollout_length"],
        "Greedy max tile": result["greedy"]["max_tile"],
        "Episodes": result["config"]["num_episodes"],
        "Double DQN": result["config"]["double_dqn"],
        "Loss": result["config"]["loss_type"],
        "n-step": result["config"]["n_step"],
        "Model type": result["config"]["model_type"],
    }
    return row

comparison_df = pd.DataFrame([
    make_comparison_row(base_result),
    make_comparison_row(enhance_result),
])

pct_cols = ["P(max tile >= 256)", "P(max tile >= 512)"]
display_df = comparison_df.copy()
for c in pct_cols:
    display_df[c] = (100.0 * display_df[c]).map(lambda x: f"{x:.1f}%")

display(display_df)
comparison_df.to_csv("comparison_table_base_vs_enhance.csv", index=False)
print("Saved comparison_table_base_vs_enhance.csv")


## 9. Detailed head-to-head interpretation


In [ ]:
def compare_and_comment(base_result, enhance_result):
    base = make_comparison_row(base_result)
    enh = make_comparison_row(enhance_result)

    print("=== HEAD-TO-HEAD SUMMARY ===")
    print(f"Base     | Greedy={base['Greedy return']:.1f} | Val={base['Validation mean return']:.1f} | Test={base['Test mean return']:.1f} | Std={base['Std']:.1f} | >=256={100*base['P(max tile >= 256)']:.1f}% | >=512={100*base['P(max tile >= 512)']:.1f}%")
    print(f"Enhance  | Greedy={enh['Greedy return']:.1f} | Val={enh['Validation mean return']:.1f} | Test={enh['Test mean return']:.1f} | Std={enh['Std']:.1f} | >=256={100*enh['P(max tile >= 256)']:.1f}% | >=512={100*enh['P(max tile >= 512)']:.1f}%")
    print()

    deltas = {
        "Greedy return": enh["Greedy return"] - base["Greedy return"],
        "Validation mean return": enh["Validation mean return"] - base["Validation mean return"],
        "Test mean return": enh["Test mean return"] - base["Test mean return"],
        "P(max tile >= 256)": enh["P(max tile >= 256)"] - base["P(max tile >= 256)"],
        "P(max tile >= 512)": enh["P(max tile >= 512)"] - base["P(max tile >= 512)"],
        "Mean max tile": enh["Mean max tile"] - base["Mean max tile"],
    }

    print("=== ENHANCE MINUS BASE ===")
    print(f"Greedy return delta: {deltas['Greedy return']:.1f}")
    print(f"Validation mean delta: {deltas['Validation mean return']:.1f}")
    print(f"Test mean delta: {deltas['Test mean return']:.1f}")
    print(f"P(max tile >= 256) delta: {100*deltas['P(max tile >= 256)']:.1f}%")
    print(f"P(max tile >= 512) delta: {100*deltas['P(max tile >= 512)']:.1f}%")
    print(f"Mean max tile delta: {deltas['Mean max tile']:.1f}")
    print()

    if deltas["Test mean return"] > 0 and deltas["P(max tile >= 256)"] >= 0 and deltas["P(max tile >= 512)"] >= 0:
        print("Interpretation: the enhanced model outperforms the base on the main fixed-seed test protocol.")
    else:
        print("Interpretation: the enhanced model does not dominate the base on every metric in this run. Try increasing the enhanced budget or training with a second training seed.")

compare_and_comment(base_result, enhance_result)


## 10. Curves: training return, validation mean return, and TD loss


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(base_result["episode_returns"], label="Base", alpha=0.8)
plt.plot(enhance_result["episode_returns"], label="Enhance", alpha=0.8)
plt.title("Episode return during training")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
if len(base_result["val_history"]) > 0:
    x_base = [x["episode"] for x in base_result["val_history"]]
    y_base = [x["mean_return"] for x in base_result["val_history"]]
    plt.plot(x_base, y_base, label="Base", marker="o")
if len(enhance_result["val_history"]) > 0:
    x_enh = [x["episode"] for x in enhance_result["val_history"]]
    y_enh = [x["mean_return"] for x in enhance_result["val_history"]]
    plt.plot(x_enh, y_enh, label="Enhance", marker="o")
plt.title("Validation mean return (fixed seeds)")
plt.xlabel("Episode")
plt.ylabel("Mean return")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
if len(base_result["loss_history"]) > 0:
    plt.plot(base_result["loss_history"], label="Base", alpha=0.7)
if len(enhance_result["loss_history"]) > 0:
    plt.plot(enhance_result["loss_history"], label="Enhance", alpha=0.7)
plt.title("TD loss")
plt.xlabel("Update step")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## 11. Show final greedy boards for qualitative comparison


In [ ]:
def show_final_board(result):
    print("=" * 70)
    print(result["config"]["name"])
    print("Greedy return:", result["greedy"]["greedy_return"])
    print("Rollout length:", result["greedy"]["rollout_length"])
    print("Max tile:", result["greedy"]["max_tile"])
    print("Final board:")
    print(result["greedy"]["last_board"])

show_final_board(base_result)
show_final_board(enhance_result)


## 12. A small ablation inside the enhanced family (optional)

This section is optional but useful for the report. It isolates which changes matter inside the enhanced pipeline:
- Base
- Base + Double DQN
- Base + Huber
- Base + Double DQN + Huber
- Full Enhance (Dueling + Residual + 3-step + Double + Huber)

Set `RUN_SMALL_ABLATION = True` only if you are willing to spend extra compute.


In [ ]:
RUN_SMALL_ABLATION = False

if RUN_SMALL_ABLATION:
    ablation_configs = []

    c1 = deepcopy(BASE_CONFIG)
    c1["name"] = "Base"

    c2 = deepcopy(BASE_CONFIG)
    c2["name"] = "Base + Double DQN"
    c2["double_dqn"] = True

    c3 = deepcopy(BASE_CONFIG)
    c3["name"] = "Base + Huber"
    c3["loss_type"] = "huber"

    c4 = deepcopy(BASE_CONFIG)
    c4["name"] = "Base + Double DQN + Huber"
    c4["double_dqn"] = True
    c4["loss_type"] = "huber"

    c5 = deepcopy(ENHANCE_CONFIG)
    c5["name"] = "Full Enhance"

    ablation_configs = [c1, c2, c3, c4, c5]
    ablation_results = [train_variant(cfg) for cfg in ablation_configs]
    ablation_df = pd.DataFrame([make_comparison_row(r) for r in ablation_results])

    display_df = ablation_df.copy()
    for c in ["P(max tile >= 256)", "P(max tile >= 512)"]:
        display_df[c] = (100.0 * display_df[c]).map(lambda x: f"{x:.1f}%")
    display(display_df)
    ablation_df.to_csv("ablation_table.csv", index=False)
    print("Saved ablation_table.csv")
else:
    print("Skipping optional ablation runs. Set RUN_SMALL_ABLATION = True to enable.")


## 13. Suggested report discussion

You can write the analysis around these points:

### Wrapper choice
Both models use the same thin OpenSpiel wrapper, so the comparison is not confounded by environment logic changes.

### Baseline
The base model is a simple MLP DQN with legal-action masking. It is easy to train and often gets decent single-rollout results, but it does not have advanced value-estimation stabilizers.

### Enhanced model
The enhanced model upgrades the learner with:
- **Double DQN** to reduce overestimation,
- **Huber loss** for robustness,
- **Dueling architecture** to separate state value from action advantage,
- **Residual MLP backbone** for deeper representation without a large CNN,
- **n-step returns** to propagate reward information faster.

### Why the enhanced model should have a better chance to dominate
This notebook intentionally gives the enhanced model a stronger architecture and a larger training budget by default. That makes it easier to obtain a clear performance gap when the practical goal is to show that the enhanced agent can outperform the base.

### What to cite from the tables
Use the comparison table as the main evidence:
- Greedy return
- Validation mean return
- Test mean return
- Test std
- P(max tile >= 256)
- P(max tile >= 512)

The fixed validation/test seeds are especially important because they make the comparison more reproducible than a single lucky rollout.
